In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"F:\COURS IA\3A\P2 courses\REINFORFCEMENT LEARNING\rl-trading-agent")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "application" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from environment.setup_env import build_envs
from agents.DRL import DRLAgent
from agents.architectures import SimplePortfolioMLP

train_env, test_env, train_df, test_df = build_envs()
print(type(train_env))
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

[*********************100%***********************]  1 of 1 completed

1. Downloading data



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (4527, 8)
2. Adding technical indicators
Successfully added technical indicators
3. Computing rolling covariance
4. Splitting train / test
Train : 2019-01-03 → 2021-12-31 (2268 rows)
Test  : 2022-01-03 → 2023-12-29 (1503 rows)
5. Creating environments
Done
<class 'environment.portfolio_env.StockPortfolioEnv'>
Train rows: 2268 | Test rows: 1503


In [ ]:
agent = DRLAgent(env=train_env)

pg_model = agent.get_model(
    "pg",
    policy=SimplePortfolioMLP,
    device="cpu",
    model_kwargs={
        "batch_size": 64,
        "lr": 1e-3,
        "action_noise": 0.01,
    },
    policy_kwargs={
        "input_shape": (7, 3),
        "portfolio_size": 3,
        "hidden_dim": 64,
        "device": "cpu",
    },
)

trained_pg = agent.train_model(pg_model, episodes=50)
trained_pg

Training PG:   2%|▏         | 1/50 [00:01<01:01,  1.25s/it, reward=129693454.3825, loss=-0.003429, pv=278418.27, steps=756]

begin_total_asset:100000
end_total_asset:278418.2688757595
Sharpe:  1.3509016586363911


Training PG:   4%|▍         | 2/50 [00:02<00:59,  1.23s/it, reward=143733844.4904, loss=-0.003430, pv=311195.26, steps=756]

begin_total_asset:100000
end_total_asset:311195.2551791737
Sharpe:  1.4715675080070278


In [ ]:
results_pg, actions_pg = DRLAgent.DRL_prediction(
    model=trained_pg,
    environment=test_env,
    online_training_period=10**9,
    lr=0,
)

results_path = OUTPUT_DIR / "results_pg.csv"
actions_path = OUTPUT_DIR / "actions_pg.csv"

results_pg.to_csv(results_path, index=False)
actions_pg.to_csv(actions_path, index=False)

print(f"Saved results to: {results_path}")
print(f"Saved actions to: {actions_path}")

print(results_pg.head())
print(actions_pg.head())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

results_df = pd.read_csv(results_path)
actions_df = pd.read_csv(actions_path)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
results_df.plot(x="date", y="portfolio_value", ax=axes[0], legend=False, title="Portfolio value on test set")
results_df.plot(x="date", y="daily_return", ax=axes[1], legend=False, title="Daily return on test set")
plt.tight_layout()
plt.show()

display(actions_df.head())

saved_plots = list((PROJECT_ROOT / "application").rglob("*.png")) + list(PROJECT_ROOT.rglob("cumulative_reward.png")) + list(PROJECT_ROOT.rglob("rewards.png"))
for plot_path in sorted({path.resolve() for path in saved_plots}):
    print(plot_path)
    display(Image(filename=str(plot_path)))